In [8]:
import pandas as pd
import re

In [9]:
import pandas as pd
import re

df = pd.read_csv('grampa.csv', index_col=0)

print(f"原始数据形状: {df.shape}")
print(f"原始列名: {list(df.columns)}")

# 1. 重命名列
rename_dict = {
    'bacterium': 'Raw_Species',
    'sequence': 'Sequence',
    'strain': 'Strain',
    'value': 'log_MIC_Value(uM)'
}
df.rename(columns=rename_dict, inplace=True)

# 清洗序列
# 氨基酸序列要清洗，保留20种标准的氨基酸字母
def clean_sequence(seq: str) -> str:
    seq = seq.upper().strip()
    standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if set(seq).issubset(standard_aa):
        return seq
    else:
        return None

df["Sequence"] = df["Sequence"].apply(clean_sequence)
df = df.dropna(subset=['Sequence'])

# 2. 生成新的ID列：database + url_source最后的数字
def extract_id(row):
    """
    从database和url_source生成ID
    例如: database='APD', url_source='http://aps.unmc.edu/AP/database/query_output.php?ID=1958'
    生成: 'APD_1958'
    """
    database = row['database']
    url_source = row['url_source']
    match = None  # 初始化match变量
    
    if 'id=DRAMP' in url_source:
        match = re.search(r'id=DRAMP(\d+)', str(url_source))
    elif 'split4' in url_source:
        match = re.search(r'id=(.*)', str(url_source))
        if match:
            return match.group(1)
        else:
            return url_source
    else:
        match = re.search(r'=(\d+)', str(url_source))
    
    if match:
        id_num = match.group(1)
        return f"{database}_{id_num}"
    else: 
        # 如果没有找到，返回原index
        return url_source

df['ID'] = df.apply(extract_id, axis=1)
df['Length'] = df['Sequence'].apply(lambda x : len(str(x)))

# 3. 重新排列列的顺序，把ID放在前面
columns_order = ['ID', 'Sequence', 'Length' , 'log_MIC_Value(uM)', 'Raw_Species',  'Strain']
df = df[[col for col in columns_order if col in df.columns]]

print(f"\n处理后的列名: {list(df.columns)}")
print(f"\n前5行数据:")
print(df.head())

# 保存
output_file = 'grampa_processed.csv'
df.to_csv(output_file, index=False)
print(f"\n已保存到 {output_file}")
print(f"总行数: {len(df)}")


原始数据形状: (51345, 12)
原始列名: ['bacterium', 'modifications', 'sequence', 'strain', 'unit', 'url_source', 'value', 'is_modified', 'has_unusual_modification', 'has_cterminal_amidation', 'datasource_has_modifications', 'database']

处理后的列名: ['ID', 'Sequence', 'Length', 'log_MIC_Value(uM)', 'Raw_Species', 'Strain']

前5行数据:
         ID                                    Sequence  Length  \
0  APD_1958                 GLPRKILCAIAKKKGKCKGPLKLVCKC      27   
1  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   
2  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   
3  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   
4  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   

   log_MIC_Value(uM)     Raw_Species     Strain  
0          -0.397940     B. subtilis        NaN  
1          -1.859819   S. agalactiae        NaN  
2           0.566150       S. aureus  ATCC29213  
3          -1.240030    S. pneumonia  ATCC49619  
4           0.566150  S. epider

In [4]:
df = pd.read_csv("final_grampa_processed.csv", dtype={"Tax_ID":str})
print(len(df))
print(df.head(5))

51345
         ID                                    Sequence  Length  \
0  APD_1958                 GLPRKILCAIAKKKGKCKGPLKLVCKC      27   
1  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   
2  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   
3  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   
4  APD_2119  GFGCPGDAYQCSEHCRALGGGRTGGYCAGPWYLGHPTCTCSF      42   

   log_MIC_Value(uM)  Tax_ID     Raw_Species              Target_Species  \
0          -0.397940    1423     B. subtilis           Bacillus subtilis   
1          -1.859819    1311   S. agalactiae    Streptococcus agalactiae   
2           0.566150  193461       S. aureus         Streptomyces aureus   
3          -1.240030    1313    S. pneumonia    Streptococcus pneumoniae   
4           0.566150    1282  S. epidermidis  Staphylococcus epidermidis   

      Strain  
0        NaN  
1        NaN  
2  ATCC29213  
3  ATCC49619  
4  ATCC12228  


In [5]:
# 统计Target_Species的不同物种数量
# 如果Target_Species是NaN，则使用Raw_Species的值

# 创建一个新的Series，用来处理NaN值
species_list = []
from_raw_species = set()  # 记录来自Raw_Species的物种

for idx, row in df.iterrows():
    if pd.isna(row['Target_Species']):
        # 如果Target_Species是NaN，使用Raw_Species
        species = row['Raw_Species']
        from_raw_species.add(species)
    else:
        # 否则使用Target_Species
        species = row['Target_Species']
    species_list.append(species)

# 统计不同物种的数量
unique_species = pd.Series(species_list).value_counts()
print(f"总共有 {len(unique_species)} 种不同的物种\n")
print("=" * 80)
print("各物种的数量统计:")
print("=" * 80)

# 设置pandas显示选项，不省略输出
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

for species, count in unique_species.items():
    if species in from_raw_species:
        print(f"{species}: {count} (来自Raw_Species, Target_Species为NaN)")
    else:
        print(f"{species}: {count}")

print("=" * 80)
print(f"总条数: {len(species_list)}")
print(f"Target_Species为NaN的物种数: {len(from_raw_species)}")



总共有 666 种不同的物种

各物种的数量统计:
Escherichia coli: 9160
Streptomyces aureus: 8956
Pseudomonas aeruginosa: 5257
Candida albicans: 2861
Bacillus subtilis: 2401
Staphylococcus epidermidis: 1451
Salmonella typhimurium: 1202
Klebsiella pneumoniae: 1158
Micrococcus luteus: 1120
Enterococcus faecalis: 1019
Acinetobacter baumannii: 690
Bacillus cereus: 488
Salmonella enterica: 469
Listeria monocytogenes: 407
Priestia megaterium: 400
Pseudomonas syringae: 370
Enterobacter cloacae: 360
Enterococcus faecium: 307
Saccharomyces cerevisiae: 284
Streptococcus mutans: 267
Streptococcus pyogenes: 254
Cryptococcus neoformans: 248
Fusarium oxysporum: 227
Proteus mirabilis: 203
Listeria innocua: 186
Candida parapsilosis: 183
Lactococcus lactis: 179
Proteus vulgaris: 173
Serratia marcescens: 171
H. lung: 170 (来自Raw_Species, Target_Species为NaN)
Streptococcus pneumoniae: 165
H. breast: 163 (来自Raw_Species, Target_Species为NaN)
H. cervical: 161 (来自Raw_Species, Target_Species为NaN)
Candida tropicalis: 155
Erwinia amylov

In [6]:
df.dropna(subset=["Tax_ID"], inplace=True)
df.to_csv("final_GRAMPA_MIC_processed.csv")
print(len(df))

48765


In [7]:
df["Target_Species"].isna().sum()

0